# 07 — Structure preservation of reconstructed SASP profiles (Table 12, Supplement S6)
4-fold held-out-entry reconstruction of every measured spot x gene entry with the full model (3 seeds) and Ring2Mean; observed vs reconstructed structure (k-means ARI at k = 3/5/7 in the observed PCA space, PCA-space R2, SASP-score correlations).
(1) The profiles are regenerated and compared with the 125 stored `.npz` profile files (observed and Ring2Mean must be identical; model profiles agree up to GPU float tolerance). (2) The metrics are recomputed from the regenerated profiles and from the stored profiles and compared with the stored metric tables. No training. The qualitative figures (Figs 3-5) are produced by `scripts/figures_structure_as_run_in_revision.py` and are not re-run here.

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, json, pandas as pd, numpy as np, torch
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability, effective_mask, score_mask
from sasp_gat.evaluation import load_model, eval_both
from sasp_gat.training import evaluate_multimask_av

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
ppi_ei = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order, device)
CKDIR = REPO / 'checkpoints' if (REPO / 'checkpoints' / 'full_avail_seed42.pt').exists() else Path(BASE) / 'revision_checkpoints_2026-10-18'
print('checkpoint dir:', CKDIR)
import sklearn; print('scikit-learn', sklearn.__version__, '(KMeans/ARI results can depend on this version)')
import torch_geometric, scipy; print('versions: torch', torch.__version__, '| pyg', torch_geometric.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__)
import warnings; warnings.filterwarnings('ignore')
RC_DIR = Path(BASE) / 'revision_reconstruction_2026-10-20'
ABL_DIR = Path(BASE) / 'revision_checkpoints_2026-10-18'
OUT_DIR = Path(BASE) / 'repro_outputs_2026-10-24'; OUT_DIR.mkdir(exist_ok=True)
from sasp_gat import structure as st
from sasp_gat.evaluation import load_all_models
from sasp_gat.baselines import train_gene_means
from sasp_gat.verify import compare_csv, report_compare, roundtrip


In [ ]:
lock = json.load(open(RC_DIR / 'package9_protocol_v1.json', encoding='utf-8'))
SL = [(c, x['slide']) for c in ('test', 'hd') for x in lock['scope']['slides'][c]]; assert len(SL) == 25
mu_tr = np.array(lock['feature_space']['train_gene_mean'], dtype=np.float64); sd_tr = np.array(lock['feature_space']['train_gene_sd'], dtype=np.float64)
mu_new = train_gene_means(train_s, AVAIL).double()
print('locked TRAIN gene mean vs recomputed (availability-aware) max |diff|: %.2e' % np.abs(mu_tr - mu_new.numpy()).max())
x_all = torch.cat([d.x for d in train_s.values()]).double().numpy()
print('locked TRAIN gene SD vs ddof=1 / ddof=0 over TRAIN spots (all genes, incl. unmeasured zero-fill):', np.abs(sd_tr - x_all.std(0, ddof=1)).max(), np.abs(sd_tr - x_all.std(0, ddof=0)).max())
unm = {}
for e in lock['reconstruction']['unmeasured_entries']:
    s_, g_ = (e if isinstance(e, (list, tuple)) else (e.get('slide', e.get('sample_id', e.get('slide_id'))), e['gene']))
    unm.setdefault(s_, set()).add(lock['feature_space']['gene_order'].index(g_))
n_meas_lock = {x['slide']: x['n_measured_genes'] for c in ('test', 'hd') for x in lock['scope']['slides'][c]}
MODELS = load_all_models(CKDIR, device, variants=['full'])
FULL_MODELS = {s: MODELS[('full', s)] for s in SEEDS}

In [ ]:
# 1) regenerate the 125 profiles in memory and compare with the stored .npz files
PROF, INFO = {}, {}
all_ds = {**test_s, **hd_s}
for coh, sid in SL:
    PROF[sid], INFO[sid] = st.reconstruct_slide(sid, all_ds[sid], AVAIL[sid], FULL_MODELS, ppi_ei, mu_new, device)
rows = []
for coh, sid in SL:
    for p in st.PROFILES:
        z = np.load(RC_DIR / 'recon_profiles_v1' / f'{sid}_{p}.npz', allow_pickle=False)
        key = 'observed_values' if p == 'observed' else 'reconstructed_values'
        assert np.array_equal(z['gene_indices'], INFO[sid]['gene_indices']) and np.array_equal(z['fold'], INFO[sid]['fold'])
        a, b = np.asarray(z[key], dtype=np.float64), PROF[sid][p].astype(np.float64)
        rows.append(dict(slide=sid, profile=p, max_abs_diff=float(np.abs(a - b).max()), identical=bool(np.array_equal(z[key], PROF[sid][p]))))
        if p == 'ring2mean': assert np.array_equal(z['fallback_mask'], INFO[sid]['r2_fallback'])
PD_ = pd.DataFrame(rows)
print(PD_.groupby('profile').agg(n=('slide', 'size'), n_identical=('identical', 'sum'), max_diff=('max_abs_diff', 'max')).to_string())
assert PD_[PD_.profile.isin(['observed', 'ring2mean'])].identical.all(), 'observed / Ring2Mean profiles must be identical'
print('model profiles: max |diff| to stored = %.2e (GPU float tolerance)' % PD_[PD_.profile.str.startswith('full')].max_abs_diff.max())

In [ ]:
# 2) metrics: from regenerated profiles and from the stored profiles; compare with the stored tables
def getter_regen(sid, p): return INFO[sid]['gene_indices'], PROF[sid][p], (INFO[sid]['r2_fallback'] if p == 'ring2mean' else None)
def getter_stored(sid, p):
    z = np.load(RC_DIR / 'recon_profiles_v1' / f'{sid}_{p}.npz', allow_pickle=False)
    return z['gene_indices'], (z['observed_values'] if p == 'observed' else z['reconstructed_values']), (z['fallback_mask'] if p == 'ring2mean' else None)
out = {}
for label, g in {'stored profiles': getter_stored, 'regenerated profiles': getter_regen}.items():
    D, CS = st.compute_metrics(SL, g, mu_tr, sd_tr, unm, n_meas_lock)
    # stored profiles reproduce the metrics exactly (rtol 1e-6). Regenerated model profiles carry GPU float32 noise (<= 2e-6 in the profiles); continuous metrics follow
    # (<= 1e-6), but ARI depends on KMeans labels and is probably sensitive to such noise (observed: HD TENX162, seed 2026, k=5; which labels changed was not inspected). A 'DIFFERENT' for the
    # regenerated profiles is therefore expected to be confined to ARI rows - the next cell lists them. The reported numbers come from the stored profiles.
    TOL = dict(rtol=1e-6, atol=1e-9) if label == 'stored profiles' else dict(rtol=1e-5, atol=1e-5)
    res = {'reconstruction_slide_metrics_v1.csv': compare_csv(roundtrip(D), RC_DIR / 'reconstruction_slide_metrics_v1.csv', **TOL),
           'reconstruction_cohort_summary_v1.csv': compare_csv(roundtrip(CS), RC_DIR / 'reconstruction_cohort_summary_v1.csv', **TOL)}
    print('--', label, D.shape, CS.shape); report_compare(res); out[label] = (D, CS)
D, CS = out['regenerated profiles']; D.to_csv(OUT_DIR / 'reconstruction_slide_metrics_v1.csv', index=False); CS.to_csv(OUT_DIR / 'reconstruction_cohort_summary_v1.csv', index=False)
pd.set_option('display.width', 220)
print(CS[CS.profile == 'full_minus_ring2'].round(4).to_string(index=False))

## Numerical-precision check of the regenerated profiles
The metrics computed from the *stored* profiles equal the stored tables exactly (above). Model profiles regenerated on a GPU carry float32 noise (<= 2e-6 on a Tesla T4). This cell checks, row by row, what that noise changes: continuous metrics (PCA R2, score correlations), all counts and all signs of full-minus-Ring2Mean must be unaffected, and every row where the **ARI** (KMeans-based, discrete; probably sensitive to such noise, the changed labels were not inspected) moves is listed explicitly.

In [ ]:
import numpy as np, pandas as pd
(Ds, CSs), (Dr, CSr) = out['stored profiles'], out['regenerated profiles']
norm = lambda s: s.astype(object).where(s.notna(), 'NA').astype(str)
def pos_align(a, b, name):
    assert a.shape == b.shape and list(a.columns) == list(b.columns), (name, a.shape, b.shape)
    num = ['value', 'seed_sd', 'median', 'q25', 'q75']
    ids = [c for c in a.columns if c not in num and not c.startswith('n_')]
    same = {c: bool((norm(a[c]).to_numpy() == norm(b[c]).to_numpy()).all()) for c in ids}
    print(name, 'identifier columns identical row by row:', same); assert all(same.values())
pos_align(Ds, Dr, 'slide metrics'); pos_align(CSs, CSr, 'cohort summary')
f = lambda s: s.astype(float).to_numpy()
dv, dsd = np.abs(f(Ds.value) - f(Dr.value)), np.abs(f(Ds.seed_sd) - f(Dr.seed_sd))
is_ari = (Dr.metric == 'ARI').to_numpy()
print('\nnon-ARI metrics: max |diff| value %.2e, seed_sd %.2e' % (np.nanmax(dv[~is_ari]), np.nanmax(dsd[~is_ari])))
assert np.nanmax(dv[~is_ari]) < 1e-5 and np.nanmax(dsd[~is_ari]) < 1e-5, 'continuous metrics changed'
mk = (Ds.profile == 'full_minus_ring2').to_numpy(); s, r = f(Ds.value)[mk], f(Dr.value)[mk]; fin = np.isfinite(s) & np.isfinite(r)
print('full_minus_ring2 rows:', int(fin.sum()), '| sign flips:', int((np.sign(s[fin]) != np.sign(r[fin])).sum()))
for c in ['n_valid_slides', 'n_comparable', 'n_full_better', 'n_equal', 'n_ring2_better']:
    a, b = f(CSs[c]), f(CSr[c]); assert ((a == b) | (np.isnan(a) & np.isnan(b))).all(), c
print('all counts (n_valid_slides, n_comparable, n_full_better, n_equal, n_ring2_better) identical')
for c in ['median', 'q25', 'q75']:
    s, r = f(CSs[c]), f(CSr[c]); m = ~((CSs.metric == 'ARI').to_numpy())
    print(f'{c:6s} non-ARI rows max |diff| {np.nanmax(np.abs(s - r)[m]):.2e} | ARI rows max |diff| {np.nanmax(np.abs(s - r)[~m]):.2e}')
print('\nARI rows that differ between stored and regenerated profiles (|diff|>1e-6):')
print(Dr.assign(value_stored=f(Ds.value), diff=dv)[dv > 1e-6][['cohort', 'slide', 'profile', 'seed', 'metric', 'k', 'value_stored', 'value', 'diff']].to_string(index=False))
cd = {c: np.abs(f(CSs[c]) - f(CSr[c])) for c in ['median', 'q25', 'q75']}; anyd = (cd['median'] > 1e-6) | (cd['q25'] > 1e-6) | (cd['q75'] > 1e-6)
print('cohort-summary rows that differ:'); print(CSr.assign(q25_stored=CSs.q25.values, q75_stored=CSs.q75.values, median_stored=CSs['median'].values)[anyd][['cohort', 'profile', 'metric', 'k', 'median_stored', 'median', 'q25_stored', 'q25', 'q75_stored', 'q75']].to_string(index=False))